In [2]:
library(MLmetrics)
library(randomForest)
set.seed(2) 

Warning message:
"le package 'MLmetrics' a été compilé avec la version R 4.2.3"

Attachement du package : 'MLmetrics'


L'objet suivant est masqué depuis 'package:base':

    Recall


Warning message:
"le package 'randomForest' a été compilé avec la version R 4.2.3"
randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.



In [3]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [4]:
data<-read.csv("train_values.csv",stringsAsFactors = T)
data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
datam<-merge(data,data_labels,by=c('building_id','building_id'))

In [5]:
ncol(datam)

[1] 40

We do not need to one hot encode categorical variables for randomForest. We will now scan for the optimal parameters n_trees and nb of features selected at each split. The function tuneRF could allow us to scan of nb of features selected at each split, but using was not optimal: we feel like it stopped too early in the search. Usual values of the parameter are sqrt(p), log2(p), ln(p) with p the amount of features.

In [7]:
n_trees <- c(10,20,100,200,500)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

  |                                                                      |   0%[1] "Number of trees used: 10 mTry: 3"
ntree      OOB      1      2      3
    1:  39.79% 62.21% 18.84% 68.92%
    2:  39.10% 64.25% 15.85% 71.27%
    3:  38.03% 62.60% 16.87% 66.88%
    4:  37.68% 62.37% 16.89% 65.85%
    5:  37.49% 62.46% 16.65% 65.58%
    6:  37.08% 61.43% 16.71% 64.54%
    7:  36.65% 61.23% 16.16% 64.29%
    8:  36.19% 60.92% 15.65% 63.90%
    9:  36.06% 61.02% 15.28% 64.08%
   10:  35.58% 60.82% 14.96% 63.28%
[1] "Best F1 Score -  10 trees - mtry 3 : 10 1"               
[2] "Best F1 Score -  10 trees - mtry 3 : 3 1"                
[3] "Best F1 Score -  10 trees - mtry 3 : 0.664396308589628 1"
  |====================================================                  |  75%[1] "Number of trees used: 10 mTry: 6"
ntree      OOB      1      2      3
    1:  37.04% 55.99% 29.21% 44.72%
    2:  36.99% 55.56% 28.35% 46.20%
    3:  36.59% 55.19% 27.61% 46.49%
    4:  36.18% 55.26% 27.10% 46.09%

ERROR: Error in xy.coords(x, y, xlabel, ylabel, log): les longueurs de 'x' et 'y' diffèrent


In [ ]:
n_trees <- 1000
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(1.5*sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat,14,16)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

  |                                                                      |   0%[1] "Number of trees used: 1000 mTry: 3"
ntree      OOB      1      2      3
    1:  39.79% 62.21% 18.84% 68.92%
    2:  39.10% 64.25% 15.85% 71.27%
    3:  38.03% 62.60% 16.87% 66.88%
    4:  37.68% 62.37% 16.89% 65.85%
    5:  37.49% 62.46% 16.65% 65.58%
    6:  37.08% 61.43% 16.71% 64.54%
    7:  36.65% 61.23% 16.16% 64.29%
    8:  36.19% 60.92% 15.65% 63.90%
    9:  36.06% 61.02% 15.28% 64.08%
   10:  35.58% 60.82% 14.96% 63.28%
   11:  35.31% 60.63% 14.27% 63.73%
   12:  35.37% 61.16% 13.84% 64.47%
   13:  34.95% 60.63% 13.50% 63.97%
   14:  35.04% 61.03% 12.93% 65.07%
   15:  34.82% 61.03% 13.02% 64.29%
   16:  34.55% 61.06% 12.63% 64.13%
   17:  34.58% 61.44% 12.54% 64.27%
   18:  34.39% 61.55% 12.40% 63.90%
   19:  34.02% 61.65% 12.21% 63.10%
   20:  33.95% 61.75% 11.87% 63.45%
   21:  33.99% 61.71% 11.25% 64.60%
   22:  34.05% 61.86% 11.04% 65.12%
   23:  33.91% 62.00% 10.98% 64.74%
   24:  33.90% 6

: 

: 

In [7]:
n_trees <- 500
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 500 mTry: 6"
ntree      OOB      1      2      3
    1:  35.25% 57.39% 26.93% 43.03%
    2:  35.58% 56.76% 26.18% 45.56%
    3:  35.20% 55.86% 25.80% 45.26%
    4:  34.80% 55.66% 24.95% 45.52%
    5:  34.25% 55.41% 24.11% 45.42%
    6:  33.82% 55.57% 23.18% 45.71%
    7:  33.37% 55.49% 22.39% 45.71%
    8:  33.00% 56.00% 21.46% 46.03%
    9:  32.46% 56.20% 20.80% 45.51%
   10:  32.13% 56.52% 20.10% 45.61%
   11:  31.77% 56.43% 19.60% 45.39%
   12:  31.36% 56.37% 19.05% 45.13%
   13:  31.03% 56.20% 18.72% 44.76%
   14:  30.72% 56.10% 18.32% 44.56%
   15:  30.55% 56.29% 18.00% 44.52%
   16:  30.45% 56.37% 17.75% 44.62%
   17:  30.25% 56.47% 17.47% 44.48%
   18:  30.06% 56.65% 17.12% 44.46%
   19:  29.87% 56.58% 16.87% 44.32%
   20:  29.76% 56.50% 16.68% 44.35%
   21:  29.55% 56.55% 16.44% 44.12%
   22:  29.46% 56.72% 16.28% 44.08%
   23:  29.39% 56.77% 16.12% 44.13%
   24:  29.29% 56.76% 16.00% 44.02%
   25:  29.26% 56.73% 15.80% 44.29%
   26:  29.14% 56.80% 15

ERROR: Error in eval(expr, envir, enclos): objet 'i' introuvable


In [8]:
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)

[1] "Best F1 Score -  500 trees - mtry 6 : 0.727787264250494 1"
  n_trees mtry        F1
1     500    6 0.7277873


In [9]:
n_trees <- 500
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(2*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 500 mTry: 12"
ntree      OOB      1      2      3
    1:  35.54% 52.20% 30.30% 39.75%
    2:  35.83% 51.03% 29.75% 41.83%
    3:  35.62% 50.72% 29.12% 42.34%
    4:  35.21% 50.09% 28.38% 42.54%
    5:  34.96% 50.27% 27.95% 42.45%
    6:  34.39% 50.12% 27.16% 42.17%
    7:  33.69% 49.76% 26.30% 41.64%
    8:  33.06% 49.51% 25.52% 41.15%
    9:  32.57% 49.51% 24.84% 40.81%
   10:  32.24% 49.83% 24.30% 40.65%
   11:  31.82% 49.43% 23.78% 40.40%
   12:  31.46% 49.65% 23.22% 40.23%
   13:  31.07% 49.66% 22.72% 39.90%
   14:  30.76% 49.55% 22.30% 39.72%
   15:  30.47% 49.77% 21.90% 39.46%
   16:  30.29% 49.81% 21.60% 39.43%
   17:  30.10% 49.84% 21.33% 39.31%
   18:  29.89% 49.94% 21.09% 39.07%
   19:  29.66% 49.64% 20.84% 38.88%
   20:  29.53% 50.01% 20.63% 38.75%
   21:  29.37% 49.89% 20.42% 38.66%
   22:  29.23% 49.75% 20.25% 38.57%
   23:  29.15% 49.93% 20.11% 38.53%
   24:  29.05% 49.87% 19.93% 38.54%
   25:  28.95% 49.97% 19.77% 38.51%
   26:  28.80% 49.92% 1

In [10]:
n_trees <- 500
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(2.5*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 500 mTry: 15"
ntree      OOB      1      2      3
    1:  35.75% 51.32% 30.61% 39.94%
    2:  35.45% 50.04% 29.94% 40.58%
    3:  35.27% 49.65% 29.33% 41.22%
    4:  34.86% 49.01% 28.53% 41.55%
    5:  34.44% 49.19% 27.76% 41.55%
    6:  33.96% 48.40% 27.26% 41.20%
    7:  33.39% 48.61% 26.50% 40.73%
    8:  32.86% 48.52% 25.73% 40.46%
    9:  32.45% 48.67% 25.15% 40.19%
   10:  31.99% 49.11% 24.53% 39.75%
   11:  31.67% 49.19% 23.99% 39.70%
   12:  31.24% 49.10% 23.49% 39.27%
   13:  30.98% 49.15% 23.05% 39.24%
   14:  30.68% 48.86% 22.63% 39.15%
   15:  30.38% 49.20% 22.27% 38.75%
   16:  30.17% 48.78% 22.01% 38.68%
   17:  30.02% 49.11% 21.80% 38.50%
   18:  29.92% 49.07% 21.58% 38.60%
   19:  29.75% 49.28% 21.33% 38.45%
   20:  29.65% 49.14% 21.15% 38.48%
   21:  29.56% 49.27% 21.05% 38.34%
   22:  29.37% 49.09% 20.77% 38.32%
   23:  29.34% 49.24% 20.67% 38.35%
   24:  29.19% 49.17% 20.48% 38.24%
   25:  29.07% 49.11% 20.33% 38.16%
   26:  29.07% 49.45% 2

In [11]:
n_trees <- 500
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(3*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 500 mTry: 18"
ntree      OOB      1      2      3
    1:  36.21% 52.11% 30.93% 40.58%
    2:  36.08% 52.15% 30.14% 41.46%
    3:  35.84% 50.86% 29.56% 42.15%
    4:  35.35% 50.51% 28.70% 42.25%
    5:  34.78% 50.35% 27.80% 42.15%
    6:  34.20% 49.95% 27.07% 41.77%
    7:  33.73% 50.22% 26.41% 41.42%
    8:  33.14% 49.61% 25.66% 41.10%
    9:  32.71% 49.67% 25.04% 40.84%
   10:  32.08% 49.27% 24.26% 40.41%
   11:  31.78% 49.54% 23.79% 40.22%
   12:  31.46% 49.53% 23.35% 40.01%
   13:  31.14% 49.61% 22.95% 39.72%
   14:  30.85% 49.61% 22.50% 39.63%
   15:  30.61% 49.67% 22.23% 39.34%
   16:  30.33% 49.49% 21.91% 39.13%
   17:  30.16% 49.60% 21.60% 39.09%
   18:  30.01% 49.57% 21.41% 38.97%
   19:  29.74% 49.33% 21.11% 38.75%
   20:  29.59% 49.35% 20.91% 38.65%
   21:  29.50% 49.26% 20.74% 38.69%
   22:  29.35% 49.30% 20.54% 38.57%
   23:  29.30% 49.44% 20.40% 38.60%
   24:  29.12% 49.69% 20.16% 38.41%
   25:  29.09% 49.57% 20.11% 38.44%
   26:  28.96% 49.50% 1

In [12]:
n_trees <- 750
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(1.5*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 750 mTry: 9"
ntree      OOB      1      2      3
    1:  36.18% 53.59% 30.12% 41.56%
    2:  35.89% 52.27% 29.36% 42.34%
    3:  35.26% 51.05% 28.33% 42.52%
    4:  34.88% 50.49% 27.58% 42.80%
    5:  34.26% 50.46% 26.75% 42.38%
    6:  33.89% 50.55% 26.11% 42.34%
    7:  33.44% 50.05% 25.41% 42.33%
    8:  32.91% 50.13% 24.50% 42.24%
    9:  32.60% 50.36% 24.04% 42.04%
   10:  32.14% 50.46% 23.35% 41.82%
   11:  31.69% 50.19% 22.92% 41.30%
   12:  31.33% 50.20% 22.36% 41.13%
   13:  30.95% 50.08% 21.94% 40.78%
   14:  30.66% 50.42% 21.46% 40.62%
   15:  30.36% 50.21% 21.10% 40.40%
   16:  30.15% 50.48% 20.71% 40.33%
   17:  29.97% 50.40% 20.43% 40.31%
   18:  29.72% 50.32% 20.13% 40.09%
   19:  29.62% 50.32% 19.97% 40.08%
   20:  29.48% 50.33% 19.70% 40.09%
   21:  29.35% 50.49% 19.48% 40.05%
   22:  29.21% 50.59% 19.28% 39.93%
   23:  29.11% 50.58% 19.15% 39.86%
   24:  28.97% 50.43% 19.02% 39.72%
   25:  28.88% 50.39% 18.92% 39.64%
   26:  28.83% 50.60% 18

In [13]:
n_trees <- 750
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(2*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 750 mTry: 12"
ntree      OOB      1      2      3
    1:  36.01% 53.65% 30.43% 40.42%
    2:  36.20% 52.28% 29.95% 42.18%
    3:  36.00% 51.61% 29.34% 42.85%
    4:  35.45% 50.97% 28.48% 42.87%
    5:  34.84% 50.16% 27.82% 42.40%
    6:  34.29% 49.78% 27.07% 42.14%
    7:  33.73% 49.76% 26.23% 41.88%
    8:  33.16% 49.32% 25.58% 41.41%
    9:  32.64% 49.40% 24.87% 41.04%
   10:  32.29% 49.50% 24.27% 40.97%
   11:  31.94% 49.79% 23.82% 40.60%
   12:  31.57% 49.81% 23.28% 40.43%
   13:  31.20% 49.64% 22.84% 40.12%
   14:  30.89% 49.80% 22.41% 39.88%
   15:  30.57% 49.49% 21.99% 39.71%
   16:  30.42% 49.76% 21.72% 39.65%
   17:  30.22% 49.65% 21.47% 39.52%
   18:  29.99% 49.78% 21.13% 39.38%
   19:  29.88% 49.82% 20.93% 39.38%
   20:  29.73% 49.94% 20.68% 39.29%
   21:  29.63% 49.77% 20.65% 39.10%
   22:  29.47% 50.04% 20.37% 39.01%
   23:  29.31% 50.14% 20.17% 38.87%
   24:  29.27% 50.03% 20.07% 38.95%
   25:  29.19% 50.17% 19.94% 38.90%
   26:  29.02% 50.07% 1

In [14]:
n_trees <- 1000
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- floor(2*sqrt(nfeat))
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
target_variable <- ncol(datam)-1 #I will remove id_variable too so I need to remove 1 col
id_variable <- match('building_id', colnames(datam))


#3.1 Take the first half of the dataset as a training data set
print(paste("Number of trees used:",n_trees,'mTry:',m_tries))
train_data <- datam[datam_idx[1:split],-c(id_variable)]

#3.2 Take the second half of the dataset as a hold out or test data set
test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]

model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,mtry=m_tries,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
#model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
#    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
#model
yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
accuracy_vec[nrow(accuracy_vec)+1,]<-c(n_trees,m_tries,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
print(paste("Best F1 Score - ",n_trees, 'trees','- mtry',m_tries,':',accuracy_vec[nrow(accuracy_vec),3],n=1))


print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv')

[1] "Number of trees used: 1000 mTry: 12"
ntree      OOB      1      2      3
    1:  36.16% 53.02% 31.24% 39.65%
    2:  35.87% 51.02% 30.81% 40.09%
    3:  35.66% 50.27% 30.08% 40.96%
    4:  35.28% 49.30% 29.58% 40.95%
    5:  34.88% 48.90% 28.93% 40.93%
    6:  34.35% 48.96% 28.23% 40.55%
    7:  33.81% 48.74% 27.53% 40.18%
    8:  33.46% 48.86% 26.95% 40.06%
    9:  32.98% 48.72% 26.36% 39.69%
   10:  32.53% 48.98% 25.67% 39.44%
   11:  32.06% 48.52% 25.16% 39.03%
   12:  31.70% 48.50% 24.72% 38.73%
   13:  31.43% 48.68% 24.26% 38.63%
   14:  31.21% 48.69% 23.94% 38.52%
   15:  30.91% 48.57% 23.53% 38.34%
   16:  30.64% 48.70% 23.24% 38.01%
   17:  30.43% 48.67% 23.01% 37.78%
   18:  30.23% 48.83% 22.75% 37.58%
   19:  30.05% 48.90% 22.50% 37.45%
   20:  29.85% 48.64% 22.33% 37.21%
   21:  29.72% 48.74% 22.13% 37.12%
   22:  29.57% 48.70% 21.85% 37.18%
   23:  29.42% 48.87% 21.65% 37.02%
   24:  29.30% 48.66% 21.50% 36.96%
   25:  29.22% 48.95% 21.37% 36.87%
   26:  29.13% 48.76% 

Other param to try: 
mtries =  floor(sqrt(nfeat)),floor(1.5*sqrt(nfeat)),floor(2*sqrt(nfeat))

n_trees = 750,1000


In [ ]:
write.csv(accuracy_vec,'randomForest_tuning_temp.csv')